# Quick calc of mean T for base and analysis to get the difference

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#from mpl_toolkits.basemap import Basemap

import scipy.stats as stats
from sklearn import linear_model
import statsmodels.api as sm
import seaborn as sns

from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt

In [2]:
# read in data for land and ocean
# input_path = '../../../Data/ERA5-global/Analysis/New/1961-2023_LAND_summer_stats.nc'
# ds_land = xr.open_dataset(input_path)

# input_path2 = '../../../Data/ERA5-global/Analysis/New/1961-2023_OCEAN_summer_stats.nc'
# ds_ocean = xr.open_dataset(input_path2)
# merge land and ocean for now
#ds_full = ds_land.merge(ds_ocean)

# read in ALL data for ocean, land, and coastal areas with all flags
input_path = '../../../Data/ERA5-global/Analysis/New/1961-2023_ALL_summer_stats.nc'
ds_full = xr.open_dataset(input_path)

# generate weights for any spatial averages
# (this is a regular grid so we can use cos(latitude))
weights = np.cos(np.deg2rad(ds_full.lat))
weights.name = "weights"



In [3]:
ds_full

<xarray.Dataset> Size: 5GB
Dimensions:       (lat: 721, lon: 1440, time: 63)
Coordinates:
  * lat           (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon           (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
  * time          (time) datetime64[ns] 504B 1961-01-01 ... 2023-01-01
Data variables:
    SummerStart   (time, lat, lon) float64 523MB ...
    SummerEnd     (time, lat, lon) float64 523MB ...
    SummerTmax    (time, lat, lon) float64 523MB ...
    SummerHeat    (time, lat, lon) float64 523MB ...
    SummerLength  (time, lat, lon) float64 523MB ...
    SummerRMSE    (time, lat, lon) float64 523MB ...
    SummerR2      (time, lat, lon) float64 523MB ...
    SummerMeanT   (time, lat, lon) float64 523MB ...
    SummerMeanHS  (time, lat, lon) float64 523MB ...
    lsm           (time, lat, lon) float64 523MB ...
    Coastal       (time, lat, lon) bool 65MB ...

In [8]:
# latitude & longitude ranges for NH and SH midlatitudes so they will be easy to adjust
nh_min = 23.5
nh_max = 70
sh_min = -23.5
sh_max = -70

# Land Temp weighted averages for the midlatitudes
# removing grid cells from averages where the mean summer temp was 0 K meaning no summer
nh_base_T = ds_full.sel(time=slice('1961','1990')).SummerMeanT.where((ds_full.lsm > 0.5) & (ds_full.SummerMeanT > 0) &
                             ((ds_full.lat >= nh_min) & (ds_full.lat <= nh_max))).weighted(weights).mean(dim=['lat','lon'])


sh_base_T = ds_full.sel(time=slice('1961','1990')).SummerMeanT.where((ds_full.lsm > 0.5) & (ds_full.SummerMeanT > 0) & 
                             ((ds_full.lat >= sh_max) & (ds_full.lat <= sh_min))).weighted(weights).mean(dim=['lat','lon'])


nh_an_T = ds_full.sel(time=slice('1990','2023')).SummerMeanT.where((ds_full.lsm > 0.5) & (ds_full.SummerMeanT > 0) &
                                                          ((ds_full.lat >= nh_min) & (ds_full.lat <= nh_max))).weighted(weights).mean(dim=['lat','lon'])

sh_an_T = ds_full.sel(time=slice('1990','2023')).SummerMeanT.where((ds_full.lsm > 0.5) & (ds_full.SummerMeanT > 0) &
                                                           ((ds_full.lat >= sh_max) & (ds_full.lat <= sh_min))).weighted(weights).mean(dim=['lat','lon'])


In [9]:
nh_an_T.mean() - nh_base_T.mean(), sh_an_T.mean() - sh_base_T.mean()

(<xarray.DataArray 'SummerMeanT' ()> Size: 8B
 array(0.32708759),
 <xarray.DataArray 'SummerMeanT' ()> Size: 8B
 array(-0.085911))